# 02: Training models A to E

Trains the five configurations from thesis Table 3.3 on MRNet under the matched protocol of
section 3.6 (weighted BCE, AdamW, cosine schedule, mixed precision where the backend supports
it, gradient accumulation over 8 examinations, early stopping on mean tuning AUC)

Every run writes `outputs/runs/<run_id>/` containing `best.pt`, `preds_tuning.csv`,
`preds_test.csv` and `log.json`. Every Chapter 4 table is computed from these files, so nothing
needs retraining later. The grid cell is resumable: re-running the notebook skips finished
runs, so an interrupted overnight session just continues where it stopped

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.config import Config, PLANES, get_device
from src import training as T

cfg = Config(smoke_test=False).make_dirs()   # smoke test - but reminder to self: make `False` after
print("Device:", get_device())

Device: mps


## 2.1 Run budget

Pick a preset. The overnight preset applies the budget cuts allowed by the implementation guide
(1 seed instead of 3, a 15-epoch cap instead of 40, model D on the sagittal plane only) so the
whole grid fits in roughly 6 to 8 hours on this Mac. Verify with the timing cell in 2.3 before
committing to it, and record every cut in thesis section 5.8.

| Preset | Seeds | Epoch cap | Runs | Rough wall time |
|---|---|---|---|---|
| `smoke` | 1 | 2 | 7 tiny runs | minutes |
| `overnight` | 1 | 15 | 13 | about 6 to 8 h |
| `full-protocol` | 3 | 40 | 45 | days, needs a CUDA GPU |

In [2]:
PRESET = "overnight"

if PRESET == "smoke":
    cfg.max_epochs, cfg.patience, SEEDS = 2, 2, [0]
    MODELS = ["A", "B", "C", "D", "E"]
    PLANES_FOR = {"A": ["sagittal"], "B": ["sagittal"], "E": ["sagittal"],
                  "C": PLANES, "D": ["sagittal"]}
elif PRESET == "overnight":
    cfg.max_epochs, cfg.patience, SEEDS = 15, 4, [0]
    MODELS = ["A", "B", "C", "D", "E"]
    PLANES_FOR = {"D": ["sagittal"]}          # guide budget cut: D on sagittal only
else:  # full-protocol
    cfg.max_epochs, cfg.patience, SEEDS = 40, 8, [0, 1, 2]
    MODELS = ["A", "B", "C", "D", "E"]
    PLANES_FOR = {}

specs = T.build_run_grid(cfg, MODELS, seeds=SEEDS, planes_for=PLANES_FOR)
print(f"{len(specs)} runs queued:")
for s in specs:
    print("  ", s.run_id)

13 runs queued:
   A_sagittal_s0
   A_coronal_s0
   A_axial_s0
   B_sagittal_s0
   B_coronal_s0
   B_axial_s0
   C_sagittal_s0
   C_coronal_s0
   C_axial_s0
   D_sagittal_s0
   E_sagittal_s0
   E_coronal_s0
   E_axial_s0


## 2.2 Sanity check: overfit a handful of exams

In [3]:
RUN_SANITY = True
if RUN_SANITY:
    for m in MODELS:
        T.overfit_check(cfg, m, n=8 if cfg.smoke_test else 20,
                        epochs=15 if cfg.smoke_test else 25)

Model A: final loss on 20 exams = 0.0228 -> PASS
Model B: final loss on 20 exams = 0.0254 -> PASS
Model C: final loss on 20 exams = 0.0227 -> PASS
Model D: final loss on 20 exams = 0.0274 -> PASS
Model E: final loss on 20 exams = 0.0227 -> PASS


## 2.3 Time one epoch and project the budget

In [4]:
RUN_TIMING = not cfg.smoke_test
if RUN_TIMING:
    est = {m: T.time_one_epoch(cfg, m) for m in MODELS}
    n_runs = {m: sum(1 for s in specs if s.model_id == m) for m in MODELS}
    total = sum(est[m] * cfg.max_epochs * 0.7 * n_runs[m] for m in MODELS) / 3600
    print(f"\nProjected grid total ≈ {total:.1f} h "
          f"(assumes early stopping around 70% of the {cfg.max_epochs}-epoch cap)")

Model A: 0.3 min/epoch (training pass only)
Model B: 1.0 min/epoch (training pass only)
Model C: 3.4 min/epoch (training pass only)
Model D: 3.2 min/epoch (training pass only)
Model E: 3.3 min/epoch (training pass only)

Projected grid total ≈ 4.8 h (assumes early stopping around 70% of the 15-epoch cap)


## 2.4 Optional: hyperparameter search (per thesis section 3.6)
But unfortunately, I do not think I have enough time for this per time budget constraint

In [5]:
RUN_SEARCH, N_TRIALS, EPOCH_CAP = False, 4, 6
if RUN_SEARCH:
    import json
    best = {m: T.random_search(cfg, m, n_trials=N_TRIALS, epoch_cap=EPOCH_CAP)
            for m in MODELS}
    (cfg.outputs_dir / "hparams.json").write_text(json.dumps(best, indent=2))
    for s in specs:
        s.hparams.update(best[s.model_id])
    print(best)

## 2.5 The full grid (logs Loading / Training epoch cycle)

In [8]:
logs = T.run_grid(cfg, specs, resume=True)

[1/13] training A_sagittal_s0 (elapsed 0.00 h)
  [A_sagittal_s0] epoch 00  loss 0.7902  tuning AUC 0.6499  (0.4 min)
  [A_sagittal_s0] epoch 01  loss 0.7665  tuning AUC 0.6835  (0.3 min)
  [A_sagittal_s0] epoch 02  loss 0.7558  tuning AUC 0.6709  (0.3 min)
  [A_sagittal_s0] epoch 03  loss 0.7400  tuning AUC 0.6988  (0.3 min)
  [A_sagittal_s0] epoch 04  loss 0.7360  tuning AUC 0.7048  (0.3 min)
  [A_sagittal_s0] epoch 05  loss 0.7258  tuning AUC 0.7019  (0.3 min)
  [A_sagittal_s0] epoch 06  loss 0.7350  tuning AUC 0.7040  (0.3 min)
  [A_sagittal_s0] epoch 07  loss 0.7251  tuning AUC 0.7139  (0.3 min)
  [A_sagittal_s0] epoch 08  loss 0.7235  tuning AUC 0.7105  (0.3 min)
  [A_sagittal_s0] epoch 09  loss 0.7236  tuning AUC 0.7112  (0.3 min)
  [A_sagittal_s0] epoch 10  loss 0.7188  tuning AUC 0.6986  (0.3 min)
  [A_sagittal_s0] epoch 11  loss 0.7167  tuning AUC 0.7079  (0.3 min)
[2/13] training A_coronal_s0 (elapsed 0.11 h)
  [A_coronal_s0] epoch 00  loss 0.7907  tuning AUC 0.5688  (0.3 min

In [9]:
import pandas as pd
from src.evaluate import collect_runs

runs = collect_runs(cfg)
cols = [c for c in ("run_id", "model_id", "plane", "seed", "best_epoch", "epochs_run",
                    "best_tuning_auc", "epoch_seconds_mean",
                    "inference_seconds_per_exam") if c in runs.columns]
display(runs[cols].round(4))

,run_id,model_id,plane,seed,best_epoch,epochs_run,best_tuning_auc,epoch_seconds_mean,inference_seconds_per_exam
0,A_axial_s0,A,axial,0,10,15,0.7436,17.4384,0.0974
1,A_coronal_s0,A,coronal,0,2,7,0.5873,17.1196,0.0966
2,A_sagittal_s0,A,sagittal,0,7,12,0.7139,17.6058,0.0992
3,B_axial_s0,B,axial,0,5,10,0.8623,67.2982,0.1176
4,B_coronal_s0,B,coronal,0,7,12,0.8546,60.8929,0.1181
5,B_sagittal_s0,B,sagittal,0,8,13,0.8189,63.0525,0.1144
6,C_axial_s0,C,axial,0,7,12,0.8508,205.9958,0.1599
7,C_coronal_s0,C,coronal,0,6,11,0.8516,181.1430,0.1552
8,C_sagittal_s0,C,sagittal,0,6,11,0.8076,184.0726,0.1588
9,D_sagittal_s0,D,sagittal,0,7,12,0.8404,191.4218,0.0549


## Run Outputs
All runs will be stored under `outputs/runs/`
<br>
`03_evaluation_and_figures.ipynb` will handle the table/figure artifacts later!